# Layer05 q10 vs q394 MFA Comparison

This notebook compares two K=1000 component-sharded MFA runs trained on the same layer 5 Gemma-2B activation shards:

- q=10: `dalg-cache/pile_gemma2b_activations/layer05_1000_10_component_sharded_mfa`
- q=394: `dalg-cache/pile_gemma2b_activations/layer05_1000_394_component_sharded_mfa`

The notebook is safe to run before the long q394 assignment artifact exists: it reports missing inputs and skips dependent comparison sections. Once both assignment files are present, rerun all cells for the full comparison.

## 1. Setup and Artifact Validation

In [1]:
from __future__ import annotations

import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

try:
    import plotly.express as px
except Exception as exc:
    px = None
    print(f"Plotly unavailable: {exc}")


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src/dalg").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")

REPO = find_repo_root()
SHARD_DIR = REPO / "dalg-cache/pile_gemma2b_activations"
RUN_Q10 = REPO / "dalg-cache/pile_gemma2b_activations/layer05_1000_10_component_sharded_mfa"
RUN_Q394 = REPO / "dalg-cache/pile_gemma2b_activations/layer05_1000_394_component_sharded_mfa"
OUT_DIR = REPO / "output/experiments/layer05_q10_q394_comparison"
ASSIGN_Q10 = OUT_DIR / "q10_assignments.pt"
ASSIGN_Q394 = OUT_DIR / "q394_assignments.pt"
SUMMARY_PATH = OUT_DIR / "model_summaries.pt"

LAYER = 5
K = 1000
NEIGHBOR_KS = [5, 10, 25, 50]
RANK_THRESHOLDS = [0.90, 0.95, 0.99]
PAIR_SAMPLE = 20000
PAIR_SEED = 0

cfg10 = json.loads((RUN_Q10 / "config.json").read_text())
cfg394 = json.loads((RUN_Q394 / "config.json").read_text())
summary_cfg = pd.DataFrame([
    {"model": "q10", **{k: cfg10.get(k) for k in ["K", "rank", "layer", "d_model", "window", "drop_prefix", "training_mode", "world_size"]}},
    {"model": "q394", **{k: cfg394.get(k) for k in ["K", "rank", "layer", "d_model", "window", "drop_prefix", "training_mode", "world_size"]}},
])
display(summary_cfg)

assert cfg10["K"] == cfg394["K"] == K
assert cfg10["layer"] == cfg394["layer"] == LAYER
assert cfg10["d_model"] == cfg394["d_model"]

artifact_status = pd.DataFrame([
    {"artifact": "q10 assignments", "path": str(ASSIGN_Q10.relative_to(REPO)), "exists": ASSIGN_Q10.exists(), "size_gb": ASSIGN_Q10.stat().st_size / 1e9 if ASSIGN_Q10.exists() else 0.0},
    {"artifact": "q394 assignments", "path": str(ASSIGN_Q394.relative_to(REPO)), "exists": ASSIGN_Q394.exists(), "size_gb": ASSIGN_Q394.stat().st_size / 1e9 if ASSIGN_Q394.exists() else 0.0},
])
display(artifact_status)
HAVE_BOTH_ASSIGNMENTS = bool(ASSIGN_Q10.exists() and ASSIGN_Q394.exists())
if not HAVE_BOTH_ASSIGNMENTS:
    display(Markdown("**Pending input:** full comparison sections are skipped until both assignment files exist. Run q394 assignments to create `output/experiments/layer05_q10_q394_comparison/q394_assignments.pt`."))

,model,K,rank,layer,d_model,window,drop_prefix,training_mode,world_size
0,q10,1000,10,5,2048,256,32,component_shard,2
1,q394,1000,394,5,2048,256,32,component_shard,2


,artifact,path,exists,size_gb
0,q10 assignments,output/experiments/layer05_q10_q394_comparison...,True,0.884278
1,q394 assignments,output/experiments/layer05_q10_q394_comparison...,False,0.000000


**Pending input:** full comparison sections are skipped until both assignment files exist. Run q394 assignments to create `output/experiments/layer05_q10_q394_comparison/q394_assignments.pt`.

## 2. Assignment Agreement

Build a contingency matrix and match clusters with the Hungarian algorithm.

In [2]:
if not HAVE_BOTH_ASSIGNMENTS:
    display(Markdown("Skipped: missing one or more assignment artifacts."))
else:
    def load_assignments(path: Path) -> dict:
        obj = torch.load(path, map_location="cpu")
        obj["assignments"] = obj["assignments"].to(torch.long)
        obj["cluster_sizes"] = obj["cluster_sizes"].to(torch.long)
        return obj

    assign10_obj = load_assignments(ASSIGN_Q10)
    assign394_obj = load_assignments(ASSIGN_Q394)
    a10 = assign10_obj["assignments"]
    a394 = assign394_obj["assignments"]
    assert a10.numel() == a394.numel(), (a10.numel(), a394.numel())
    assert int(assign10_obj["K"]) == int(assign394_obj["K"]) == K
    N = a10.numel()
    print(f"Loaded {N:,} token assignments")

    contingency = torch.bincount(a10 * K + a394, minlength=K*K).reshape(K, K).cpu().numpy().astype(np.int64)
    raw_same_id_agreement = float((a10 == a394).float().mean())
    row_ind, col_ind = linear_sum_assignment(-contingency)
    match_q10_to_q394 = np.full(K, -1, dtype=np.int64)
    match_q10_to_q394[row_ind] = col_ind
    matched_count = int(contingency[row_ind, col_ind].sum())
    matched_agreement = matched_count / N
    row_counts = contingency.sum(axis=1)
    col_counts = contingency.sum(axis=0)

    def entropy_from_counts(counts):
        p = counts[counts > 0].astype(np.float64)
        p = p / p.sum()
        return float(-(p * np.log(p)).sum())

    H10 = entropy_from_counts(row_counts)
    H394 = entropy_from_counts(col_counts)
    MI = 0.0
    nz_i, nz_j = np.nonzero(contingency)
    for i, j in zip(nz_i, nz_j):
        n_ij = contingency[i, j]
        MI += (n_ij / N) * math.log((n_ij * N) / (row_counts[i] * col_counts[j]))
    NMI = MI / math.sqrt(H10 * H394)

    assignment_summary = pd.DataFrame([
        {"metric": "tokens", "value": N},
        {"metric": "same numeric cluster id agreement", "value": raw_same_id_agreement},
        {"metric": "Hungarian matched agreement", "value": matched_agreement},
        {"metric": "matched token count", "value": matched_count},
        {"metric": "NMI", "value": NMI},
        {"metric": "nonempty q10 clusters", "value": int((assign10_obj["cluster_sizes"] > 0).sum())},
        {"metric": "nonempty q394 clusters", "value": int((assign394_obj["cluster_sizes"] > 0).sum())},
    ])
    display(assignment_summary)

Skipped: missing one or more assignment artifacts.

In [3]:
if not HAVE_BOTH_ASSIGNMENTS:
    display(Markdown("Skipped: missing one or more assignment artifacts."))
else:
    matched_pairs = pd.DataFrame({
        "q10_cluster": np.arange(K),
        "q394_cluster": match_q10_to_q394,
        "shared_tokens": contingency[np.arange(K), match_q10_to_q394],
        "q10_size": row_counts,
        "q394_size": col_counts[match_q10_to_q394],
    })
    matched_pairs["q10_recall_in_match"] = matched_pairs["shared_tokens"] / np.maximum(matched_pairs["q10_size"], 1)
    matched_pairs["q394_precision_in_match"] = matched_pairs["shared_tokens"] / np.maximum(matched_pairs["q394_size"], 1)
    matched_pairs["jaccard"] = matched_pairs["shared_tokens"] / np.maximum(matched_pairs["q10_size"] + matched_pairs["q394_size"] - matched_pairs["shared_tokens"], 1)
    display(matched_pairs.sort_values("shared_tokens", ascending=False).head(20))
    display(matched_pairs[["shared_tokens", "q10_recall_in_match", "q394_precision_in_match", "jaccard"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
    if px is not None:
        px.histogram(matched_pairs, x="jaccard", nbins=50, title="Hungarian Matched Cluster Jaccard Distribution").show()

Skipped: missing one or more assignment artifacts.

## 3. Assignment-Based Neighbor Overlap

In [4]:
if not HAVE_BOTH_ASSIGNMENTS:
    display(Markdown("Skipped: missing one or more assignment artifacts."))
else:
    C10 = contingency.astype(np.float64)
    co10 = C10 @ C10.T
    co394 = C10.T @ C10
    np.fill_diagonal(co10, -np.inf)
    np.fill_diagonal(co394, -np.inf)

    def topk_indices(score_mat, k):
        idx = np.argpartition(-score_mat, kth=k-1, axis=1)[:, :k]
        vals = np.take_along_axis(score_mat, idx, axis=1)
        order = np.argsort(-vals, axis=1)
        return np.take_along_axis(idx, order, axis=1)

    rows = []
    for k in NEIGHBOR_KS:
        nn10 = topk_indices(co10, k)
        nn394 = topk_indices(co394, k)
        overlaps = []
        for c in range(K):
            mapped_q10_neighbors = set(match_q10_to_q394[nn10[c]].tolist())
            q394_neighbors = set(nn394[match_q10_to_q394[c]].tolist())
            overlaps.append(len(mapped_q10_neighbors & q394_neighbors) / k)
        rows.append({"k": k, "mean_overlap": float(np.mean(overlaps)), "median_overlap": float(np.median(overlaps)), "p10": float(np.quantile(overlaps, 0.1)), "p90": float(np.quantile(overlaps, 0.9))})
    assignment_neighbor_overlap = pd.DataFrame(rows)
    display(assignment_neighbor_overlap)
    if px is not None:
        px.line(assignment_neighbor_overlap, x="k", y=["mean_overlap", "median_overlap"], markers=True, title="Assignment-Neighbor Overlap After Matching").show()

Skipped: missing one or more assignment artifacts.

## 4. Centroid-Neighborhood Stability

In [5]:
if not HAVE_BOTH_ASSIGNMENTS:
    display(Markdown("Skipped: missing one or more assignment artifacts."))
else:
    from dalg.models.mfa import load_mfa

    def summarize_model(run_dir: Path, name: str, *, active_threshold: float = 0.01, basis_sample: int = 1000) -> dict:
        print(f"Loading {name}: {run_dir}")
        model = load_mfa(run_dir / "mfa_model.pt", map_location="cpu")
        model.eval()
        with torch.no_grad():
            mu = model.mu.detach().float().cpu()
            scale = torch.nn.functional.softplus(model.scale_rho.detach()).float().cpu()
            dir_raw = model.dir_raw.detach().float().cpu()
            dir_norm = dir_raw.norm(dim=1, keepdim=True).clamp_min(float(getattr(model, "_eps", 1e-8)))
            sample_ids = torch.linspace(0, int(model.K) - 1, steps=min(basis_sample, int(model.K))).round().long().unique()
            bases = {}
            for cid in sample_ids.tolist():
                s = scale[cid]
                active = s >= max(float(s.max()) * active_threshold, 1e-8)
                if int(active.sum()) == 0:
                    active[int(s.argmax())] = True
                Wc = (dir_raw[cid] / dir_norm[cid])[:, active] * s[active][None, :]
                Q, _ = torch.linalg.qr(Wc, mode="reduced")
                bases[int(cid)] = Q.cpu()
        out = {"mu": mu, "scale": scale, "q": int(model.q), "K": int(model.K), "D": int(model.D), "sample_bases": bases}
        del model, dir_raw, dir_norm
        return out

    if SUMMARY_PATH.exists():
        summaries = torch.load(SUMMARY_PATH, map_location="cpu")
        print(f"Loaded cached summaries: {SUMMARY_PATH}")
    else:
        summaries = {"q10": summarize_model(RUN_Q10, "q10"), "q394": summarize_model(RUN_Q394, "q394")}
        torch.save(summaries, SUMMARY_PATH)
        print(f"Saved summaries: {SUMMARY_PATH}")

    mu10 = summaries["q10"]["mu"].numpy()
    mu394 = summaries["q394"]["mu"].numpy()
    print(mu10.shape, mu394.shape)

Skipped: missing one or more assignment artifacts.

In [6]:
if not HAVE_BOTH_ASSIGNMENTS:
    display(Markdown("Skipped: missing one or more assignment artifacts."))
else:
    def centroid_neighbors(mu: np.ndarray, metric: str, k_max: int) -> np.ndarray:
        if metric == "cosine":
            x = mu / np.maximum(np.linalg.norm(mu, axis=1, keepdims=True), 1e-12)
            dist = 1.0 - x @ x.T
        elif metric == "euclidean":
            dist = cdist(mu, mu, metric="euclidean")
        else:
            raise ValueError(metric)
        np.fill_diagonal(dist, np.inf)
        idx = np.argpartition(dist, kth=k_max-1, axis=1)[:, :k_max]
        vals = np.take_along_axis(dist, idx, axis=1)
        order = np.argsort(vals, axis=1)
        return np.take_along_axis(idx, order, axis=1)

    centroid_rows = []
    for metric in ["cosine", "euclidean"]:
        nn10 = centroid_neighbors(mu10, metric, max(NEIGHBOR_KS))
        nn394 = centroid_neighbors(mu394, metric, max(NEIGHBOR_KS))
        for k in NEIGHBOR_KS:
            overlaps = []
            for c in range(K):
                mapped_neighbors = set(match_q10_to_q394[nn10[c, :k]].tolist())
                target_neighbors = set(nn394[match_q10_to_q394[c], :k].tolist())
                overlaps.append(len(mapped_neighbors & target_neighbors) / k)
            centroid_rows.append({"metric": metric, "k": k, "mean_overlap": float(np.mean(overlaps)), "median_overlap": float(np.median(overlaps)), "p10": float(np.quantile(overlaps, 0.1)), "p90": float(np.quantile(overlaps, 0.9))})
    centroid_overlap = pd.DataFrame(centroid_rows)
    display(centroid_overlap)
    if px is not None:
        px.line(centroid_overlap, x="k", y="mean_overlap", color="metric", markers=True, title="Centroid kNN Overlap After Hungarian Matching").show()

Skipped: missing one or more assignment artifacts.

## 5. Loading Effective Rank

In [7]:
if not HAVE_BOTH_ASSIGNMENTS:
    display(Markdown("Skipped: missing one or more assignment artifacts."))
else:
    def loading_rank_table(scale: torch.Tensor, name: str) -> pd.DataFrame:
        scale_np = scale.numpy()
        energy = (scale_np ** 2).astype(np.float64)
        total = np.maximum(energy.sum(axis=1, keepdims=True), 1e-30)
        sorted_energy = -np.sort(-energy, axis=1)
        cum = np.cumsum(sorted_energy, axis=1) / total
        rows = []
        pr = (energy.sum(axis=1) ** 2) / np.maximum((energy ** 2).sum(axis=1), 1e-30)
        for thr in RANK_THRESHOLDS:
            ranks = (cum < thr).sum(axis=1) + 1
            rows.append({"model": name, "rank_metric": f"directions_for_{thr:.2f}_energy", "mean": ranks.mean(), "median": np.median(ranks), "p10": np.quantile(ranks, 0.1), "p90": np.quantile(ranks, 0.9), "min": ranks.min(), "max": ranks.max()})
        active = (scale_np >= scale_np.max(axis=1, keepdims=True) * 0.01).sum(axis=1)
        rows.append({"model": name, "rank_metric": "participation_ratio", "mean": pr.mean(), "median": np.median(pr), "p10": np.quantile(pr, 0.1), "p90": np.quantile(pr, 0.9), "min": pr.min(), "max": pr.max()})
        rows.append({"model": name, "rank_metric": "directions_above_1pct_of_max_scale", "mean": active.mean(), "median": np.median(active), "p10": np.quantile(active, 0.1), "p90": np.quantile(active, 0.9), "min": active.min(), "max": active.max()})
        return pd.DataFrame(rows)

    rank_summary = pd.concat([loading_rank_table(summaries["q10"]["scale"], "q10"), loading_rank_table(summaries["q394"]["scale"], "q394")], ignore_index=True)
    display(rank_summary)
    if px is not None:
        for name in ["q10", "q394"]:
            scale = summaries[name]["scale"].numpy()
            pr = ((scale ** 2).sum(axis=1) ** 2) / np.maximum((scale ** 4).sum(axis=1), 1e-30)
            px.histogram(pd.DataFrame({"participation_ratio": pr}), x="participation_ratio", nbins=60, title=f"{name} loading participation ratio").show()

Skipped: missing one or more assignment artifacts.

## 6. Covariance/Subspace Orthogonality Within Each Model

In [8]:
if not HAVE_BOTH_ASSIGNMENTS:
    display(Markdown("Skipped: missing one or more assignment artifacts."))
else:
    def sample_subspace_overlaps(bases: dict[int, torch.Tensor], *, n_pairs: int, seed: int) -> np.ndarray:
        rng = random.Random(seed)
        ids = list(bases.keys())
        vals = []
        for _ in range(n_pairs):
            a, b = rng.sample(ids, 2)
            Qa, Qb = bases[a], bases[b]
            denom = max(1, min(Qa.shape[1], Qb.shape[1]))
            vals.append(float((Qa.T @ Qb).pow(2).sum().item() / denom))
        return np.asarray(vals, dtype=np.float64)

    orth_rows = []
    orth_values = {}
    for name in ["q10", "q394"]:
        vals = sample_subspace_overlaps(summaries[name]["sample_bases"], n_pairs=PAIR_SAMPLE, seed=PAIR_SEED)
        orth_values[name] = vals
        orth_rows.append({"model": name, "sampled_pairs": len(vals), "mean_overlap": vals.mean(), "median_overlap": np.median(vals), "p10": np.quantile(vals, 0.1), "p90": np.quantile(vals, 0.9), "p99": np.quantile(vals, 0.99)})
    orth_summary = pd.DataFrame(orth_rows)
    display(orth_summary)
    if px is not None:
        plot_df = pd.DataFrame({"overlap": np.concatenate([orth_values["q10"], orth_values["q394"]]), "model": ["q10"] * len(orth_values["q10"]) + ["q394"] * len(orth_values["q394"])})
        px.histogram(plot_df, x="overlap", color="model", nbins=80, barmode="overlay", opacity=0.65, title="Within-Model Active Subspace Overlap").show()

Skipped: missing one or more assignment artifacts.

## 7. Compact Conclusion

In [9]:
if not HAVE_BOTH_ASSIGNMENTS:
    display(Markdown("Full conclusion pending q394 assignment artifact."))
else:
    conclusion = pd.DataFrame([
        {"analysis": "assignment", "metric": "Hungarian matched agreement", "value": matched_agreement},
        {"analysis": "assignment", "metric": "NMI", "value": NMI},
        {"analysis": "assignment-neighbor", "metric": "mean overlap @10", "value": float(assignment_neighbor_overlap.loc[assignment_neighbor_overlap["k"] == 10, "mean_overlap"].iloc[0])},
        {"analysis": "centroid-neighbor cosine", "metric": "mean overlap @10", "value": float(centroid_overlap[(centroid_overlap["metric"] == "cosine") & (centroid_overlap["k"] == 10)]["mean_overlap"].iloc[0])},
        {"analysis": "q394 loading", "metric": "median participation ratio", "value": float(rank_summary[(rank_summary["model"] == "q394") & (rank_summary["rank_metric"] == "participation_ratio")]["median"].iloc[0])},
        {"analysis": "q394 subspace", "metric": "median pair overlap", "value": float(orth_summary[orth_summary["model"] == "q394"]["median_overlap"].iloc[0])},
    ])
    display(conclusion)

Full conclusion pending q394 assignment artifact.